# Feedback-Loop Bias in AI Recommendation Systems

This notebook demonstrates how a naive recommendation system creates a self-reinforcing feedback loop that leads to popularity bias and reduced content diversity.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from simulator import run_simulation, calculate_gini, calculate_diversity

# Set aesthetic style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]

## 1. Running Simulations

We will run two simulations:
1. **Naive System**: Recommendations based purely on current popularity.
2. **Mitigated System**: Epsilon-Greedy exploration (10% random exploration) to break the loop.

In [ ]:
params = {
    "num_items": 200,
    "num_users": 100,
    "cycles": 50,
    "k": 5,
    "feedback_loop_strength": 0.7  # High bias factor
}

print("Running Naive Simulation...")
naive_history, naive_items = run_simulation(**params, mitigation=False)

print("Running Mitigated Simulation (Epsilon-Greedy)...")
mitigated_history, mitigated_items = run_simulation(**params, mitigation=True)

## 2. Analyzing Bias Metrics

We track two metrics:
*   **Gini Coefficient**: Measures inequality in exposure. A value of 1 means one item gets all exposure; 0 means perfect equality.
*   **Aggregate Diversity**: The total number of unique items that were recommended at least once.

In [ ]:
def extract_metrics(history):
    ginis = []
    diversities = []
    for snapshot in history:
        stats = snapshot['item_stats']
        exposures = [s[1] for s in stats]
        ginis.append(calculate_gini(exposures))
        diversities.append(calculate_diversity(stats))
    return ginis, diversities

naive_gini, naive_div = extract_metrics(naive_history)
mitigated_gini, mitigated_div = extract_metrics(mitigated_history)

fig, ax = plt.subplots(1, 2, figsize=(15, 5))

ax[0].plot(naive_gini, label="Naive (No Mitigation)", color='crimson', lw=2)
ax[0].plot(mitigated_gini, label="Mitigated (Exploration)", color='teal', lw=2)
ax[0].set_title("Exposure Inequality (Gini Coefficient)")
ax[0].set_xlabel("Simulation Cycle")
ax[0].set_ylabel("Gini Index")
ax[0].legend()

ax[1].plot(naive_div, label="Naive", color='crimson', lw=2)
ax[1].plot(mitigated_div, label="Mitigated", color='teal', lw=2)
ax[1].set_title("Aggregate Diversity Over Time")
ax[1].set_xlabel("Simulation Cycle")
ax[1].set_ylabel("Unique Items Exposed")
ax[1].legend()

plt.tight_layout()
plt.show()

## 3. Final Exposure Distribution

Notice how in the Naive system, a few items "winner-take-all" while others are never seen.

In [ ]:
naive_exposures = sorted([i.exposure for i in naive_items], reverse=True)
mitigated_exposures = sorted([i.exposure for i in mitigated_items], reverse=True)

plt.figure(figsize=(10, 6))
plt.bar(range(len(naive_exposures)), naive_exposures, alpha=0.5, label="Naive", color='red')
plt.bar(range(len(mitigated_exposures)), mitigated_exposures, alpha=0.5, label="Mitigated", color='blue')
plt.title("Final Item Exposure Distribution (Sorted)")
plt.xlabel("Item Rank (by exposure)")
plt.ylabel("Total Exposure Count")
plt.yscale('log')
plt.legend()
plt.show()